# Pré-processamento dos dados

---

## 1. Carregando e separando os dados

In [1]:
RANDOM_SEED = 1

In [2]:
import pandas as pd

leads = pd.read_csv('data/churn_aplicacao_leads_final.csv')
pedidos = pd.read_csv('data/churn_comportamental_pedidos_final.csv')

total = leads.merge(right=pedidos, on='id_cliente', how='left')

Vamos treinar dois modelos: um **comportamental** e um de **análise de leads**.

No modelo comportamental, o conjunto de treinamento será referente aos clientes que tem pedidos correspondentes na tabela de pedidos, enquanto que o conjunto de treinamento do modelo de análise de leads conterá apenas dados de clientes que nunca fizeram um pedido antes. Nesse último caso, nenhum atributo da tabela de pedidos será relevante, então podemos também removê-los dos atributos disponíveis.

Com base nisso, separamos os conjuntos usando essas regras:

In [3]:
leads_raw: pd.DataFrame = total[total.id_pedido.isna()]           # Clientes que nunca pediram
comportamental_raw: pd.DataFrame = total[total.id_pedido.notna()] # Clientes que tem pedidos

leads_raw = leads_raw.drop(['id_pedido', 'id_cliente', 'valor', 'produtos', 'data_entrega'], axis='columns') # Removendo atributos referentes a pedidos

Por fim, vamos definir uma porcentagem para a divisão entre treino + validação e teste:

In [4]:
TRAIN_VAL_RATIO = 0.8
TEST_RATIO      = 0.2

### 1.1. Separando o conjunto de dados `leads_raw`

Sem levar os pedidos em consideração, temos um conjunto de dados que não apresenta uma relação de temporalidade relevante, então a sua divisão segue de modo padrão: faremos uma divisão estratificada no atributo-alvo, o `churned`:

In [5]:
from sklearn.model_selection import train_test_split

leads_raw_X, leads_raw_y = leads_raw, leads_raw.pop('churned')

leads_raw_train_val_X, leads_raw_test_X, leads_raw_train_val_y, leads_raw_test_y = train_test_split(
    leads_raw_X,
    leads_raw_y,
    train_size=TRAIN_VAL_RATIO,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED,
    stratify=leads_raw_y
)

LEADS = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': leads_raw_train_val_X,
        'y': leads_raw_train_val_y,
    },
    'test': {
        'X': leads_raw_test_X,
        'y': leads_raw_test_y,
    },
})

print(f'Tamanho do conjunto de treino + validação: {LEADS.train_val.X.shape[0]} linhas x {LEADS.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {LEADS.test.X.shape[0]} linhas x {LEADS.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 800 linhas x 17 colunas
Tamanho do conjunto de teste:              200 linhas x 17 colunas


### 1.2. Separando o conjunto de dados `comportamental_raw`

Como o conjunto de dados comportamental leva em consideração os pedidos dos clientes, temos uma série temporal, como discutido no processo de EDA.

Para separar os dados agora, é necessário evitar que o futuro seja usado como treinamento para testes no passado, então, devemos ordenar os dados com base na data. A partir disso, basta escolher uma data de corte e separar os dois conjuntos adequadamente.

Queremos manter a mesma proporção de amostras de teste que o conjunto de análise de leads, então usaremos `TEST_RATIO` do intervalo total (90 dias) para definir a janela de corte:

In [34]:
TEST_TIMEFRAME = pd.to_timedelta(90 * TEST_RATIO, 'days')

Agora, basta separar ambos conjuntos:

In [41]:
comportamental_raw.data_entrega = comportamental_raw.data_entrega.apply(pd.to_datetime)

ordered = comportamental_raw.sort_values(by='data_entrega')

date_threshold = pd.to_datetime(ordered.data_entrega.max()) - TEST_TIMEFRAME

comportamental_raw_train_val = ordered[ordered.data_entrega < date_threshold]
comportamental_raw_test = ordered[ordered.data_entrega >= date_threshold]

COMPORTAMENTAL = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': comportamental_raw_train_val,
        'y': comportamental_raw_train_val.pop('churned'),
    },
    'test': {
        'X': comportamental_raw_test,
        'y': comportamental_raw_test.pop('churned'),
    },
})

print(f'Tamanho do conjunto de treino + validação: {COMPORTAMENTAL.train_val.X.shape[0]} linhas x {COMPORTAMENTAL.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {COMPORTAMENTAL.test.X.shape[0]}  linhas x {COMPORTAMENTAL.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 3554 linhas x 22 colunas
Tamanho do conjunto de teste:              887  linhas x 22 colunas


Verificando as proporções finais e os intervalos de tempo a serem utilizados:

In [36]:
print('Porcentagem da divisão dos dados comportamentais:')

print(f'{100 * COMPORTAMENTAL.train_val.X.shape[0] / comportamental_raw.shape[0]:.2f}% para treino e validação')
print(f'       |-> Intervalo de tempo: {90 - TEST_TIMEFRAME.days} dias')

print(f'{100 * COMPORTAMENTAL.test.X.shape[0] / comportamental_raw.shape[0]:.2f}% para teste')
print(f'       |-> Intervalo de tempo: {TEST_TIMEFRAME.days} dias')

Porcentagem da divisão dos dados comportamentais:
80.03% para treino e validação
       |-> Intervalo de tempo: 72 dias
19.97% para teste
       |-> Intervalo de tempo: 18 dias


---